⚠️ **Gemini Parse Error** — response could not be parsed as a valid notebook.
Raw output preserved below for manual recovery.

In [ ]:
{
  "cells": [
    {
      "cell_type": "markdown",
      "metadata": {},
      "source": [
        "# ODI to Databricks Migration: TRG_EMP_INSERT\n",
        "\n",
        "**Conversion Timestamp:** 2024-07-30 12:00:00\n",
        "\n",
        "**Description:** This notebook performs a full load insertion into the `trg_emp` table based on data from the `employees` table. The original ODI process involved a direct insert.\n",
        "\n",
        "**SCEN_TASK_NO Mapping:**\n",
        "- **{10}**: Session initialization (represented by widget creation and display)\n",
        "- **{20}**: Pre-processing (no explicit SQL, handled conceptually)\n",
        "- **{30}**: Target table preparation (converted to `DELETE FROM` for full load)\n",
        "- **{INSERT Statement}**: Data insertion into target table."
      ]
    },
    {
      "cell_type": "code",
      "metadata": {},
      "source": [
        "dbutils.widgets.text(\"ETL_JOB_TYPE\", \"\", \"ETL Job Type\")\n",
        "dbutils.widgets.text(\"DATASOURCE_NUM_ID\", \"1\", \"Datasource Number ID\")\n",
        "dbutils.widgets.text(\"ETL_PROC_WID\", \"-1\", \"ETL Process ID\")\n",
        "dbutils.widgets.text(\"ODI_SESS_NO\", \"-1\", \"ODI Session Number\")"
      ]
    },
    {
      "cell_type": "markdown",
      "metadata": {},
      "source": [
        "# ETL Parameters"
      ]
    },
    {
      "cell_type": "code",
      "metadata": {},
      "source": [
        "-- MAGIC %sql\n",
        "SELECT\n",
        "  '${ETL_JOB_TYPE}' AS ETL_JOB_TYPE,\n",
        "  '${DATASOURCE_NUM_ID}' AS DATASOURCE_NUM_ID,\n",
        "  '${ETL_PROC_WID}' AS ETL_PROC_WID,\n",
        "  '${ODI_SESS_NO}' AS ODI_SESS_NO;"
      ]
    },
    {
      "cell_type": "markdown",
      "metadata": {},
      "source": [
        "# Target Table Load\n",
        "\n",
        "## SCEN_TASK_NO {10}, {20}, {30}: Prepare Target Table for Full Load\n",
        "This step prepares the `trg_emp` table for a full load by deleting all existing records. This corresponds to a common full load strategy where the target is cleared before insertion. The original ODI tasks {10}, {20}, {30} would typically encompass such preparation."
      ]
    },
    {
      "cell_type": "code",
      "metadata": {},
      "source": [
        "-- MAGIC %sql\n",
        "DELETE FROM workspace.hr.trg_emp;"
      ]
    },
    {
      "cell_type": "markdown",
      "metadata": {},
      "source": [
        "## Insert Data into Target Table\n",
        "\n",
        "This step inserts data from the `employees` table into the `trg_emp` target table. This directly converts the `INSERT` statement found in the ODI source, which was part of the final data load logic."
      ]
    },
    {
      "cell_type": "code",
      "metadata": {},
      "source": [
        "-- MAGIC %sql\n",
        "INSERT INTO workspace.hr.trg_emp\n",
        "  (\n",
        "    employee_id ,\n",
        "    first_name ,\n",
        "    last_name ,\n",
        "    email ,\n",
        "    phone_number ,\n",
        "    hire_date ,\n",
        "    job_id ,\n",
        "    salary ,\n",
        "    commission_pct ,\n",
        "    manager_id ,\n",
        "    department_id\n",
        "  )\n",
        "SELECT\n",
        "  EMPLOYEES.employee_id ,\n",
        "  EMPLOYEES.first_name ,\n",
        "  EMPLOYEES.last_name ,\n",
        "  EMPLOYEES.email ,\n",
        "  EMPLOYEES.phone_number ,\n",
        "  EMPLOYEES.hire_date ,\n",
        "  EMPLOYEES.job_id ,\n",
        "  EMPLOYEES.salary ,\n",
        "  EMPLOYEES.commission_pct ,\
        "  EMPLOYEES.manager_id ,\n",
        "  EMPLOYEES.department_id\n",
        "FROM\n",
        "  workspace.hr.employees AS EMPLOYEES;"
      ]
    },
    {
      "cell_type": "markdown",
      "metadata": {},
      "source": [
        "# Validation"
      ]
    },
    {
      "cell_type": "code",
      "metadata": {},
      "source": [
        "-- MAGIC %sql\n",
        "SELECT COUNT(*) AS target_record_count FROM workspace.hr.trg_emp;"
      ]
    },
    {
      "cell_type": "code",
      "metadata": {},
      "source": [
        "-- MAGIC %sql\n",
        "SELECT\n",
        "    employee_id,\n",
        "    first_name,\n",
        "    last_name,\n",
        "    hire_date\n",
        "FROM workspace.hr.trg_emp\n",
        "ORDER BY hire_date DESC\n",
        "LIMIT 10;"
      ]
    },
    {
      "cell_type": "markdown",
      "metadata": {},
      "source": [
        "# Conversion Notes and Manual Actions Required\n",
        "\n",
        "1.  **Schema and Table Naming**: Oracle schema `HR` has been converted to `workspace.hr` and table names to lowercase (e.g., `TRG_EMP` to `trg_emp`).\n",
        "2.  **Oracle Hints Removal**: The `/*+ APPEND PARALLEL */` hint has been removed as it is not applicable to Databricks Delta Lake.\n",
        "3.  **Full Load Strategy**: Given that the source provided only an `INSERT` statement for `TRG_EMP` preceded by `SCEN_TASK_NO {30}` (which is often a cleanup step in ODI for full loads), a `DELETE FROM workspace.hr.trg_emp` statement has been added prior to the `INSERT`. This ensures a clean full load of the target table. If the intention was an incremental load or selective insert, this `DELETE` statement might need adjustment based on the actual ODI logic for tasks {10}, {20}, {30}.\n",
        "4.  **Data Types**: Assuming the `TRG_EMP` table already exists in Databricks with compatible Spark SQL data types (e.g., Oracle `NUMBER` to `BIGINT`/`DECIMAL`, `VARCHAR2` to `STRING`, `DATE`/`TIMESTAMP` to `TIMESTAMP`). If the table creation DDL were provided, a specific type mapping would be applied as per the system prompt.\n",
        "5.  **Widget Parameters**: Standard ETL parameters (`ETL_JOB_TYPE`, `DATASOURCE_NUM_ID`, `ETL_PROC_WID`, `ODI_SESS_NO`) have been created as `dbutils` widgets. While not directly used in this simple `INSERT` query, they are part of the standard ODI to Databricks migration framework."
      ]
    }
  ],
  "metadata": {
    "kernelspec": {
      "display_name": "Python 3",
      "language": "python",
      "name": "python3"
    },
    "language_info": {
      "codemirror_mode": {
        "name": "python",
        "version": 3
      },
      "file_extension": ".py",
      "mimetype": "text/x-python",
      "name": "python",
      "nbconvert_exporter": "python",
      "pygments_lexer": "ipython3",
      "version": "3.10.12"
    }
  },
  "nbformat": 4,
  "nbformat_minor": 5
}